In [2]:
import pandas as pd
import yaml
from pathlib import Path

root_folder = Path(r"/Users/panyue/Desktop/final_data/1_crop_management_data")

# only keep files whose name contains this text
name_filter = "crop_registration_basic"

In [3]:
wanted_columns = [
    "ID_all",
    "year",
    "crop",
    "variety",
    "date_planting",
    "date_harvest"
]

output_dir = Path("/Users/panyue/PycharmProjects/wofost_example_test/input") / "agro"
output_dir.mkdir(parents=True, exist_ok=True)

excel_files = [
    f for f in root_folder.rglob("*.xlsx")
    if not f.name.startswith("~$")
    and name_filter.lower() in f.name.lower()
]

for file in excel_files:
    try:
        df = pd.read_excel(file)

        # clean column names
        df.columns = df.columns.astype(str).str.strip()

        # optional: standardize common variants
        df = df.rename(columns={
            "Yield": "yield",
            "YIELD": "yield",
            "yield ": "yield",
            " date_planting": "date_planting",
            " date_harvest": "date_harvest"
        })

        missing = [col for col in wanted_columns if col not in df.columns]
        if missing:
            print(f"Skipped {file.name}: missing columns {missing}")
            print("Columns found:", df.columns.tolist())
            continue

        df = df[wanted_columns].copy()

        # clean dates
        for col in ["date_planting", "date_harvest"]:
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
            df[col] = df[col].dt.strftime("%Y-%m-%d")

        field_name = None
        if "ID_field" in df.columns:
            non_null_fields = df["ID_field"].dropna()
            if not non_null_fields.empty:
                field_name = str(non_null_fields.iloc[0])
        if not field_name:
            field_name = file.stem

        # Create WOFOST-compatible agro management structure
        agro_management = []
        
        for idx, row in df.iterrows():
            # Use planting date as the key for AgroManagement
            planting_date = row['date_planting']
            
            agro_entry = {
                planting_date: {
                    "CropCalendar": {
                        "crop_start_date": row['date_planting'],
                        "crop_start_type": "sowing",
                        "crop_end_date": row['date_harvest'],
                        "crop_end_type": "harvest",
                        "crop_name": row['crop'].lower(),
                        "variety_name": row['variety'],
                        "max_duration": 365
                    },
                    "StateEvents": None,
                    "TimedEvents": {}
                }
            }
            agro_management.append(agro_entry)
        
        # Create the final YAML structure
        output_data = {
            "AgroManagement": agro_management,
            "Version": "1.0"
        }

        output_path = output_dir / f"agro_{field_name}.yaml"
        with open(output_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(output_data, f, sort_keys=False, default_flow_style=False)
        print(f"Wrote {output_path}")

    except Exception as e:
        print(f"Skipped {file}: {e}")

Wrote /Users/panyue/PycharmProjects/wofost_example_test/input/agro/agro_crop_registration_basic_DR14.yaml
Skipped /Users/panyue/Desktop/final_data/1_crop_management_data/ZW12/crop_registration_basic_ZW12.xlsx: 'float' object has no attribute 'lower'
Wrote /Users/panyue/PycharmProjects/wofost_example_test/input/agro/agro_crop_registration_basic_DR13.yaml
Wrote /Users/panyue/PycharmProjects/wofost_example_test/input/agro/agro_crop_registration_basic_ZW15.yaml
Wrote /Users/panyue/PycharmProjects/wofost_example_test/input/agro/agro_crop_registration_basic_ZW22.yaml
Wrote /Users/panyue/PycharmProjects/wofost_example_test/input/agro/agro_crop_registration_basic_DR12.yaml
Wrote /Users/panyue/PycharmProjects/wofost_example_test/input/agro/agro_crop_registration_basic_ZW14.yaml
Wrote /Users/panyue/PycharmProjects/wofost_example_test/input/agro/agro_crop_registration_basic_DR15.yaml
Skipped /Users/panyue/Desktop/final_data/1_crop_management_data/ZW13/crop_registration_basic_ZW13.xlsx: 'float' ob

/Users/panyue/Library/Python/3.14/lib/python/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
/Users/panyue/Library/Python/3.14/lib/python/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


ModuleNotFoundError: No module named 'pcse'

In [5]:

# Extract all potato cultivars from raw data
import pandas as pd
from pathlib import Path

root_folder = Path(r"/Users/panyue/Desktop/final_data/1_crop_management_data")

# Find all crop_registration_basic files
excel_files = [
    f for f in root_folder.rglob("*.xlsx")
    if not f.name.startswith("~$")
    and "crop_registration_basic" in f.name.lower()
]

print(f"Found {len(excel_files)} files to process\n")

# Collect all potato data
all_potato_data = []

for file in excel_files:
    try:
        df = pd.read_excel(file)
        df.columns = df.columns.astype(str).str.strip()
        
        # Filter for potato rows
        if 'crop' in df.columns:
            potato_df = df[df['crop'].str.lower().str.contains('potato', na=False)].copy()
            
            if len(potato_df) > 0:
                print(f"\n{file.name}:")
                print(f"  Found {len(potato_df)} potato records")
                all_potato_data.append(potato_df)
                
    except Exception as e:
        print(f"Error processing {file.name}: {e}")

# Combine all potato data
if all_potato_data:
    combined_potatoes = pd.concat(all_potato_data, ignore_index=True)
    
    # Get unique varieties
    if 'variety' in combined_potatoes.columns:
        unique_varieties = combined_potatoes['variety'].dropna().unique()
        print(f"\n{'='*80}")
        print(f"Total unique potato varieties found: {len(unique_varieties)}")
        print(f"{'='*80}")
        
        for variety in sorted(unique_varieties):
            count = (combined_potatoes['variety'] == variety).sum()
            print(f"  {variety}: {count} occurrences")
        
        # Save to CSV
        output_path = Path("/Users/panyue/PycharmProjects/wofost_example_test/potato_varieties.csv")
        combined_potatoes.to_csv(output_path, index=False)
        print(f"\nFull potato data saved to: {output_path}")
else:
    print("No potato records found!")


Found 40 files to process


crop_registration_basic_DR14.xlsx:
  Found 3 potato records

crop_registration_basic_ZW12.xlsx:
  Found 4 potato records

crop_registration_basic_DR13.xlsx:
  Found 4 potato records

crop_registration_basic_ZW15.xlsx:
  Found 2 potato records

crop_registration_basic_ZW22.xlsx:
  Found 2 potato records

crop_registration_basic_DR12.xlsx:
  Found 4 potato records

crop_registration_basic_ZW14.xlsx:
  Found 3 potato records

crop_registration_basic_DR15.xlsx:
  Found 3 potato records

crop_registration_basic_ZW13.xlsx:
  Found 3 potato records

crop_registration_basic_DR08.xlsx:
  Found 4 potato records

crop_registration_basic_ZW09.xlsx:
  Found 3 potato records

crop_registration_basic_DR06.xlsx:
  Found 1 potato records

crop_registration_basic_ZW07.xlsx:
  Found 4 potato records

crop_registration_basic_DR01.xlsx:
  Found 3 potato records

crop_registration_basic_ZW06.xlsx:
  Found 3 potato records

crop_registration_basic_ZW01.xlsx:
  Found 2 potato recor

/Users/panyue/Library/Python/3.14/lib/python/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
/Users/panyue/Library/Python/3.14/lib/python/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
